<a href="https://colab.research.google.com/github/sinchugowda887/AI---Projects/blob/main/Ask%20my%20pdf%20Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q faiss-cpu pypdf sentence-transformers transformers torch

In [9]:
from google.colab import files

uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

print("Using:", pdf_file)

Saving Moscow-Creative-Resume-Template.pdf to Moscow-Creative-Resume-Template (4).pdf
Using: Moscow-Creative-Resume-Template (4).pdf


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
import re

# ======================
# READ PDF
# ======================
reader = PdfReader(pdf_file)

text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("PDF loaded ✔")

# ======================
# CHUNK TEXT (SAFE)
# ======================
chunks = re.split(r'\n\s*\n', text)
chunks = [c.strip() for c in chunks if len(c.strip()) > 30]

print("Chunks:", len(chunks))

# ======================
# EMBEDDINGS MODEL
# ======================
model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(chunks)

# ======================
# FAISS INDEX
# ======================
index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

print("Vector DB ready ✔")

# ======================
# LLM (FIXED FOR YOUR ERROR)
# ======================
llm = pipeline("text-generation", model="google/flan-t5-base")

print("\n🚀 BOT READY (type 'exit' to stop)\n")

# ======================
# CHAT LOOP
# ======================
while True:

    query = input("Ask: ")

    if query.lower() == "exit":
        break

    print("Processing your question...")

    # ======================
    # RETRIEVE TOP CHUNKS
    # ======================
    q_vec = model.encode([query])
    D, I = index.search(np.array(q_vec), k=3)

    context = "\n".join([chunks[i] for i in I[0]])[:1500]

    # ======================
    # PROMPT
    # ======================
    prompt = f"""
Use only the context below to answer.

Context:
{context}

Question:
{query}

Answer:
"""

    # ======================
    # GENERATE ANSWER
    # ======================
    result = llm(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )

    print("\nAnswer:\n")
    print(result[0]["generated_text"])
    print("\n" + "-"*50)

PDF loaded ✔
Chunks: 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB ready ✔


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM',


🚀 BOT READY (type 'exit' to stop)

Ask: what is the education background?
Processing your question...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:


Use only the context below to answer.

Context:
1515 Pacific Ave, Los Angeles, CA 90291, United States
(541) 754-3010 · email@email.com
MICHELLE LOPEZ, Fashion Designer
Expert Fashion Designer with 11+ years’ experience in women’s 
high-end shoes. Launched product lines for Chanel and Gucci. 
Designs showcased in Elle and Vogue. Attained recognition of 
top seller fashionista in 2017.
Details Nationality American
Place of birth San Antonio
Driving license Full
Employment History Senior Fashion Designer  at Escada, Milan
January 2017 — July 2021
Functioned as the lead designer for the 2019 women’s winter collection team and supervised 
seasonal conceptualization and design of women’s accessories, which included belts and 
bags.
• Designed attractive fashion items that coincided with the brand’s look.
• Ran the whole product design process, from primary market research, mood board 
development to sketching and design to producing the finished product.
• Contributed to the conc